---
# 01. EDA — 채용 플랫폼 로그 데이터 탐색
---


원본 노트북 2개(원본 EDA / AAR 분석 노트북) 내용을 01_eda / 02_preprocessing / 03_analysis로 나눠서 정리했다. df.head()처럼 원본 로그 행이 그대로 보이는 출력이나 DB 계정 정보는 빼거나 가림 처리함.

## 1. 환경 설정 & DB 연결 확인
---

### 1-1. 라이브러리 환경 설정

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import koreanize_matplotlib
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()
warnings.filterwarnings('ignore')

### 1-2. DB 연결

In [ ]:
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_PORT = os.getenv('DB_PORT', '3306')
DB_NAME = os.getenv('DB_NAME', 'your_db')

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4"
)

# DB 계정 정보는 .env 파일로 분리 (레포에는 .env.example만 포함)
pd.read_sql("SELECT 1 AS ok", engine)


### 1-3. 테이블 목록

In [ ]:
query = """
SHOW tables;
"""
pd.read_sql(query, engine)

In [ ]:
# 전체 테이블 컬럼 한번에 확인
tables = ['tbl_application', 'tbl_company', 'tbl_company_address', 'tbl_company_fund',
          'tbl_job', 'tbl_job_address', 'tbl_job_bookmark', 'tbl_log_2022', 'tbl_log_2023']

for t in tables:
    print(f"\n=== {t} ===")
    display(pd.read_sql(f"DESCRIBE {t};", engine))

### 1-4. 각 테이블 행 수 확인

In [ ]:
query = """
SELECT '2022' AS log_year, COUNT(*) AS row_cnt FROM tbl_log_2022
UNION ALL
SELECT '2023', COUNT(*) FROM tbl_log_2023;
"""

pd.read_sql(query, engine)

## 2. 데이터 로드 & 기본 탐색
---

### 2-1. 두 테이블 합치기 (tbl_log_2022 / tbl_log_2023)

- tbl_log_2022, log_2023 UNION ALL
- 에러 응답(400, 404, 500 등)은 정상 유저 행동이 아니므로 제외
- resp_code 200, 302만 포함

In [ ]:
query = """
SELECT uid, URL, ts, date, resp_code, method
FROM tbl_log_2022
WHERE resp_code IN ('200', '302')
UNION ALL
SELECT uid, URL, ts, date, resp_code, method
FROM tbl_log_2023
WHERE resp_code IN ('200', '302')
"""

df = pd.read_sql(query, engine)
df['ts'] = pd.to_datetime(df['ts'], format='mixed', utc=True)
df['log_date'] = pd.to_datetime(df['log_date'])

In [ ]:
df.shape

In [ ]:
# df.head()  # 원본 로그라 여기선 생략함

### 2-2. 기본 정보 확인

In [ ]:
df.info()

### 2-3. 기간 & 유저 수 확인

In [ ]:
print(f"로그 기간: {df['log_date'].min().date()} ~ {df['log_date'].max().date()}")
print(f"유니크 유저수: {df['uid'].nunique():,}명")
print()
print(df['log_date'].dt.year.value_counts().sort_index())

### 2-4. URL 파악

- URL 컬럼의 구조를 파악 -> 파싱 전략 수립
- 실제 분석은 전체 데이터 기준으로 진행 예정

In [ ]:
# 유니크 URL 전체 패턴 파악
url_patterns = (
    df['req_url']
    .str.split('?').str[0]
    .value_counts()
    .reset_index()
)
url_patterns.columns = ['url_path', 'count']
print(f"URL 패턴 수: {len(url_patterns):,}")
url_patterns.head(50)

In [ ]:
# - Acquistion

# - Activation

# - Retention

In [ ]:
step3_done = df[df['req_url'].str.contains('EVT_SIGNUP_DONE', case=False, na=False)]
print(f"EVT_SIGNUP_DONE 고유 유저 수: {step3_done['uid'].nunique():,}")